imports

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
import pandas as pd
import numpy as np
import plotly.express as px
from gencost.crosswalk import Crosswalk
from gencost.waterfall import DataBySubplant
import seaborn as sns
from matplotlib import pyplot as plt
import pandera as pa
#from pandera import Column, DataFrameSchema, Check, Index


warnings.simplefilter(action="once")

In [ ]:
pd.set_option('display.max_columns',None)

set up data source objects

In [ ]:
xwalk = Crosswalk()
# self to be able to copy / paste from waterfall.py for dev ease
self = DataBySubplant(xwalk)

check on error

In [ ]:
df_before = self._exa_by_prime()

In [ ]:
df_before

In [ ]:
df_before.query('plant_id_eia == 7873 & pf_subplant_id == 1',engine='python')

In [ ]:
df_before.query('plant_id_eia == 1001 & pf_subplant_id == 1',engine='python')

In [ ]:
df = self.merge_all(clean=True)

In [ ]:
df

In [ ]:
df.query('real_capex > 0')

In [ ]:
df['disagg_gross_check'] = (df[
                                    [
                                        "biofuel_gross_mwh",
                                        "coal_gross_mwh",
                                        "natural_gas_gross_mwh",
                                        "other_gross_mwh",
                                        "other_gas_gross_mwh",
                                        "petroleum_gross_mwh",
                                        "petroleum_coke_gross_mwh",
                                    ]
                                ].sum(axis=1)
                        )


In [ ]:
df.dtypes

In [ ]:
df.query('gross_generation_mwh == disagg_gross_check')

In [ ]:
df['check'] =  pd.Series(
                        np.isclose(
                            (df[
                                    [
                                        "biofuel_gross_mwh",
                                        "coal_gross_mwh",
                                        "natural_gas_gross_mwh",
                                        "other_gross_mwh",
                                        "other_gas_gross_mwh",
                                        "petroleum_gross_mwh",
                                        "petroleum_coke_gross_mwh",
                                    ]
                                ].sum(axis=1)
                                
                            ),
                            df["gross_generation_mwh"],
                            rtol=0.1,
                        ),
                        index=df.index)

In [ ]:
self.validate_merge_all_results(merged_all_df)

In [ ]:
failed_checks = df.query('check == False')

In [ ]:
failed_checks

In [ ]:
df['diff'] = df['disagg_gross_check'] - df['gross_generation_mwh']


In [ ]:
df.query('gross_generation_mwh != disagg_gross_check')

In [ ]:
cols = ["biofuel_gross_mwh","coal_gross_mwh","natural_gas_gross_mwh","other_gross_mwh","other_gas_gross_mwh","petroleum_gross_mwh","petroleum_coke_gross_mwh"]

In [ ]:
for col in cols:
    df[col] = np.where(df[col].isnull(),0,df[col])

In [ ]:
df.query('opex.isnull() & capex.isnull()')

In [ ]:
df.query('')

try pandera - df schema set up

In [ ]:
df

In [ ]:
dtypes = df.dtypes.reset_index()

In [ ]:
dtypes.to_csv('dtypes.csv')

In [ ]:
schema = (pa.DataFrameSchema(columns={"plant_id_eia": pa.Column(int)},
checks=[pa.Check(lambda df: df['gross_generation_mwh'] > df['net_generation_mwh']),
pa.Check(lambda df: df['capex'] > df['opex'])],
index=pa.Index(int),
strict=False,
coerce=True))

In [ ]:
schema.validate(df)